In [23]:
import os
import ast
import datetime
import pandas as pd
import numpy as np
from price_parser import Price
from currency_converter import CurrencyConverter

In [2]:
def folder_init(folder_name):
    """Create a folder if it does exist in the project directory."""
    if not os.path.exists(folder_name):
        os.mkdir(folder_name)

In [3]:
def write_dfs(oscar_df, imdb_df):
    """Write the dataframes to the filtered data folder."""
    data_folder = 'filtered/'
    folder_init(data_folder)
    oscar_df.to_csv(data_folder + 'oscar_filtered.csv')
    imdb_df.to_csv(data_folder + 'imdb_filtered.csv')

In [4]:
def write_joined(oscar_with_imdb, imdb_non_oscar):
    """Write the joined dataframes to the filtered data folder."""
    data_folder = 'filtered/'
    folder_init(data_folder)
    oscar_with_imdb.to_csv(data_folder +  'oscar_with_imdb.csv')
    imdb_non_oscar.to_csv(data_folder + 'imdb_non_oscar.csv')

In [5]:
def read_data(file1, file2):
    """Read CSV data for two data frames."""
    folder_name = 'filtered/'
    return (pd.read_csv(folder_name + file1), pd.read_csv(folder_name + file2))

In [15]:
def round_value(x):
    """
    Round values from the budget and gross columns.
    Check for any NA or existing USD symbols.
    Return a float representing the nearest currency value.
    """
    if x is None or pd.isna(x):
        # NA can dropped later
        return None
    else:
        if type(x) == str:
            # split branches in case resolution is not left to right
                if x[0] == '$':
                    x = x[1:].replace(',', '')
        x = float(x)

    if x > 100_000_000:
        # nearest ten million
        return round(x, -7)
    elif x > 10_000_000:
        # nearest hundred thousand
        return round(x, -5)
    elif x > 100_000:
        # nearest thousand
        return round(x, -3)
    elif x <= 0:
        # negative or zero
        return None
    

In [22]:
def convert_date(release_date):
    """Convert a date to a datetime format it's if not already datetime."""
    if isinstance(release_date, datetime.datetime):
        return release_date

    if isinstance(release_date, int):
        return datetime(release_date, 1, 1)

    try:
        return datetime.datetime.fromisoformat(release_date)
    except ValueError:
        # not valid datetime format
        return 0

In [21]:
def normalize_budget(budget, release_date, converter):
    """
    Using a map of currencies, parse the currency symbol of a film's budget.
    If a currency is found, use the historical_amount helper function.
    Return the historical amount in USD.
    """
    currencies = {
        '$': 'USD',
        '£': 'GBP',
        '¥': 'YEN',
        '₩': 'KRW',
        '£': 'GBP',
        '¥': 'JPY',
        '₩': 'KRW',
        '₪': 'ILS',
        '₱': 'PHP',
        '₹': 'INR',
        'CA$': 'CAD',
        'CN¥': 'CNY',
        '€': 'EUR',
        'R$': 'BRL',
        'NZ$': 'NZD',
        'NT$': 'TWD',
        'MX$': 'MXN',
        'HK$': 'HKD',
        'CA$': 'CAD',
        'A$': 'AUD'
    }

    # assume USD if budget is already float
    if type(budget) == float:
        return budget
        
    # parse price
    if type(budget) == str:
        space_idx = budget.find(' ')
        budget = budget[:space_idx]
        
    parsed = Price.fromstring(budget)
    amount = parsed.amount_float
    currency = parsed.currency
    
    if not currency:
        return 0
    
    if currency in currencies:
        currency = currencies[currency]
        
    # convert to USD using release year
    historical_amount = get_historical_rate(int(amount), currency, 'USD', release_date, converter)

    return historical_amount

In [20]:
def get_historical_rate(amount, from_currency, to_currency, release_date, converter):
    """
    Use the Currency Converter library to convert an amount between two currencies.
    Uses the currency exchange rate at a film's release date if the exchange rate is found.
    """
    try:
        date_time_obj = convert_date(release_date)
        rate = converter.convert(amount, from_currency, to_currency, date=date_time_obj)
        return rate
    except Exception as e:
        # something went wrong with historical currency conversion
        return 0

In [19]:
def get_languages(row):
    """Use the AST library to parse the languages."""
    """Returns the first language found as the primary film language according to the dataset information."""
    lang_list = ast.literal_eval(row)
    found_lang = ''
    for lang in lang_list:
        if lang != 'None':
            found_lang = lang
            break
            
    return found_lang